# PaySim Feature Engineering

## Standalone Feature Engineering Notebook

This notebook isolates the feature-engineering stage of the PaySim fraud-detection project.

It converts raw PaySim transactions into the **same 19 model-ready features used by the current Isolation Forest experiment**.

This notebook does not train the model. It focuses on:
- loading raw PaySim data
- validating the raw data
- creating engineered features
- checking the feature contract
- documenting every feature
- validating numerical quality
- exporting engineered features and metadata

### Model inputs

The final model input contains 19 numerical features.

### Deliberately excluded columns

- `isFraud` — ground truth used separately for evaluation
- `isFlaggedFraud` — not used as a model input
- `nameOrig` — account identifier
- `nameDest` — account identifier

In [1]:
# ============================================================
# BLOCK 1 — ENVIRONMENT CHECK
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 90)
print("BLOCK 1 — FEATURE ENGINEERING ENVIRONMENT")
print("=" * 90)

print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("Environment ready.")

BLOCK 1 — FEATURE ENGINEERING ENVIRONMENT
NumPy  : 2.2.6
Pandas : 2.3.3
Environment ready.


In [2]:
# ============================================================
# BLOCK 2 — CONFIGURATION
# ============================================================

DATA_PATH = Path(
    r".\data\PS_20174392719_1491204439457_log.csv"
)

# Consistent with the current Isolation Forest experiment.
MAX_ROWS = 500_000

OUTPUT_DIR = Path(
    "feature_engineering_output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("BLOCK 2 — CONFIGURATION")
print("=" * 90)

print("Dataset :", DATA_PATH.resolve())
print("Rows    :", f"{MAX_ROWS:,}")
print("Output  :", OUTPUT_DIR.resolve())

BLOCK 2 — CONFIGURATION
Dataset : D:\Real-Time-Financial-Fraud-Detection-Pipeline-\Datapreprocessing\data\PS_20174392719_1491204439457_log.csv
Rows    : 500,000
Output  : D:\Real-Time-Financial-Fraud-Detection-Pipeline-\Datapreprocessing\feature_engineering_output


In [3]:
# ============================================================
# BLOCK 3 — LOAD RAW PAYSIM
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"PaySim file not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(
    DATA_PATH,
    nrows=MAX_ROWS,
    low_memory=False
)

print("=" * 90)
print("BLOCK 3 — RAW DATA LOADED")
print("=" * 90)

print("Rows    :", f"{len(df):,}")
print("Columns :", len(df.columns))

print("\nRaw columns:")
for i, col in enumerate(
    df.columns,
    start=1
):
    print(f"{i:02d}. {col}")

print("\nPreview:")
display(df.head())

FileNotFoundError: PaySim file not found: D:\Real-Time-Financial-Fraud-Detection-Pipeline-\Datapreprocessing\data\PS_20174392719_1491204439457_log.csv

In [ ]:
# ============================================================
# BLOCK 4 — RAW DATA QUALITY
# ============================================================

quality = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing": df.isna().sum().values,
    "missing_%": (
        df.isna().sum().values / len(df) * 100
    ).round(4)
})

print("=" * 90)
print("BLOCK 4 — RAW DATA QUALITY")
print("=" * 90)

display(quality)

print(
    "\nDuplicate rows:",
    f"{df.duplicated().sum():,}"
)

print(
    "Fraud rows:",
    f"{int(df['isFraud'].sum()):,}"
)

print(
    "Fraud rate:",
    f"{df['isFraud'].mean() * 100:.4f}%"
)

## Feature Engineering Logic

The transformations below match the current Isolation Forest notebook.

### Origin account

`oldbalanceOrg` is the origin/sender balance before a transaction.

`newbalanceOrig` is the origin/sender balance after a transaction.

```text
orig_balance_change
= oldbalanceOrg - newbalanceOrig
```

### Destination account

`oldbalanceDest` is the destination/receiver balance before a transaction.

`newbalanceDest` is the destination/receiver balance after a transaction.

```text
dest_balance_change
= newbalanceDest - oldbalanceDest
```

### Balance discrepancy

```text
orig_balance_error
= amount - orig_balance_change

dest_balance_error
= amount - dest_balance_change
```

### Account-state indicators

The notebook creates binary features indicating zero balances before or after the transaction.

### Amount transformation

```text
log_amount = log1p(amount)
```

### Transaction type

`type` is one-hot encoded into:

```text
type_CASH_IN
type_CASH_OUT
type_DEBIT
type_PAYMENT
type_TRANSFER
```

In [ ]:
# ============================================================
# BLOCK 5 — FEATURE ENGINEERING
# ============================================================

work = df.copy()

# 1. Origin-account balance movement
work["orig_balance_change"] = (
    work["oldbalanceOrg"]
    - work["newbalanceOrig"]
)

# 2. Destination-account balance movement
work["dest_balance_change"] = (
    work["newbalanceDest"]
    - work["oldbalanceDest"]
)

# 3. Origin balance discrepancy
work["orig_balance_error"] = (
    work["amount"]
    - work["orig_balance_change"]
)

# 4. Destination balance discrepancy
work["dest_balance_error"] = (
    work["amount"]
    - work["dest_balance_change"]
)

# 5. Zero-balance indicators
work["orig_zero_after"] = (
    work["newbalanceOrig"] == 0
).astype(int)

work["dest_zero_before"] = (
    work["oldbalanceDest"] == 0
).astype(int)

work["dest_zero_after"] = (
    work["newbalanceDest"] == 0
).astype(int)

# 6. Log-transformed transaction amount
work["log_amount"] = np.log1p(
    work["amount"]
)

# 7. One-hot transaction type
work = pd.get_dummies(
    work,
    columns=["type"],
    prefix="type",
    dtype=int
)

# 8. Exact feature order used by the current Isolation Forest
feature_columns = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "orig_balance_change",
    "dest_balance_change",
    "orig_balance_error",
    "dest_balance_error",
    "orig_zero_after",
    "dest_zero_before",
    "dest_zero_after",
    "log_amount",
]

type_columns = sorted(
    [
        col
        for col in work.columns
        if col.startswith("type_")
    ]
)

feature_columns.extend(
    type_columns
)

# 9. Separate model input and ground truth
X = work[
    feature_columns
].astype(np.float32)

y = work[
    "isFraud"
].astype(int)

print("=" * 90)
print("BLOCK 5 — FEATURE ENGINEERING COMPLETE")
print("=" * 90)

print(
    "Rows     :",
    f"{len(X):,}"
)

print(
    "Features :",
    X.shape[1]
)

print("\nFeatures:")
for i, feature in enumerate(
    feature_columns,
    start=1
):
    print(f"{i:02d}. {feature}")

print("\nPreview:")
display(X.head())

In [ ]:
# ============================================================
# BLOCK 6 — FEATURE CONTRACT CHECK
# ============================================================

forbidden_columns = [
    "isFraud",
    "isFlaggedFraud",
    "nameOrig",
    "nameDest"
]

present_forbidden = [
    col
    for col in forbidden_columns
    if col in X.columns
]

print("=" * 90)
print("BLOCK 6 — FEATURE CONTRACT CHECK")
print("=" * 90)

for col in forbidden_columns:
    status = "FOUND" if col in X.columns else "EXCLUDED"
    print(f"{col:18s} -> {status}")

if present_forbidden:
    raise ValueError(
        f"Forbidden columns found in X: {present_forbidden}"
    )

print("\nPASS — no forbidden columns are present in X.")

In [ ]:
# ============================================================
# BLOCK 7 — NUMERICAL VALIDATION
# ============================================================

nan_count = int(
    X.isna().sum().sum()
)

inf_count = int(
    np.isinf(
        X.to_numpy()
    ).sum()
)

print("=" * 90)
print("BLOCK 7 — NUMERICAL VALIDATION")
print("=" * 90)

print("NaN values :", nan_count)
print("Inf values :", inf_count)

if nan_count != 0:
    raise ValueError(
        "NaN values detected."
    )

if inf_count != 0:
    raise ValueError(
        "Infinite values detected."
    )

print("\nPASS — feature matrix contains no NaN or Inf values.")

In [ ]:
# ============================================================
# BLOCK 8 — FEATURE DICTIONARY
# ============================================================

feature_dictionary = pd.DataFrame([
    {
        "Feature": "step",
        "Source": "step",
        "Type": "Numerical",
        "Transformation": "None",
        "Meaning": "PaySim simulation time step"
    },
    {
        "Feature": "amount",
        "Source": "amount",
        "Type": "Numerical",
        "Transformation": "None",
        "Meaning": "Transaction amount"
    },
    {
        "Feature": "oldbalanceOrg",
        "Source": "oldbalanceOrg",
        "Type": "Numerical",
        "Transformation": "None",
        "Meaning": "Origin balance before transaction"
    },
    {
        "Feature": "newbalanceOrig",
        "Source": "newbalanceOrig",
        "Type": "Numerical",
        "Transformation": "None",
        "Meaning": "Origin balance after transaction"
    },
    {
        "Feature": "oldbalanceDest",
        "Source": "oldbalanceDest",
        "Type": "Numerical",
        "Transformation": "None",
        "Meaning": "Destination balance before transaction"
    },
    {
        "Feature": "newbalanceDest",
        "Source": "newbalanceDest",
        "Type": "Numerical",
        "Transformation": "None",
        "Meaning": "Destination balance after transaction"
    },
    {
        "Feature": "orig_balance_change",
        "Source": "oldbalanceOrg + newbalanceOrig",
        "Type": "Derived numerical",
        "Transformation": "oldbalanceOrg - newbalanceOrig",
        "Meaning": "Observed origin-side balance reduction"
    },
    {
        "Feature": "dest_balance_change",
        "Source": "newbalanceDest + oldbalanceDest",
        "Type": "Derived numerical",
        "Transformation": "newbalanceDest - oldbalanceDest",
        "Meaning": "Observed destination-side balance movement"
    },
    {
        "Feature": "orig_balance_error",
        "Source": "amount + orig_balance_change",
        "Type": "Derived numerical",
        "Transformation": "amount - orig_balance_change",
        "Meaning": "Difference between amount and origin balance movement"
    },
    {
        "Feature": "dest_balance_error",
        "Source": "amount + dest_balance_change",
        "Type": "Derived numerical",
        "Transformation": "amount - dest_balance_change",
        "Meaning": "Difference between amount and destination balance movement"
    },
    {
        "Feature": "orig_zero_after",
        "Source": "newbalanceOrig",
        "Type": "Binary",
        "Transformation": "(newbalanceOrig == 0).astype(int)",
        "Meaning": "Origin balance becomes zero"
    },
    {
        "Feature": "dest_zero_before",
        "Source": "oldbalanceDest",
        "Type": "Binary",
        "Transformation": "(oldbalanceDest == 0).astype(int)",
        "Meaning": "Destination balance is zero before transaction"
    },
    {
        "Feature": "dest_zero_after",
        "Source": "newbalanceDest",
        "Type": "Binary",
        "Transformation": "(newbalanceDest == 0).astype(int)",
        "Meaning": "Destination balance is zero after transaction"
    },
    {
        "Feature": "log_amount",
        "Source": "amount",
        "Type": "Numerical",
        "Transformation": "np.log1p(amount)",
        "Meaning": "Log-transformed transaction amount"
    },
])

for category in type_columns:
    name = category.replace(
        "type_",
        ""
    )

    feature_dictionary = pd.concat(
        [
            feature_dictionary,
            pd.DataFrame([{
                "Feature": category,
                "Source": "type",
                "Type": "Binary",
                "Transformation": f"One-hot indicator for {name}",
                "Meaning": f"Transaction type is {name}"
            }])
        ],
        ignore_index=True
    )

print("=" * 90)
print("BLOCK 8 — FEATURE DICTIONARY")
print("=" * 90)

display(feature_dictionary)

In [ ]:
# ============================================================
# BLOCK 9 — SINGLE TRANSACTION EXAMPLE
# ============================================================

example_raw = df.head(1)

print("=" * 90)
print("BLOCK 9 — RAW TRANSACTION")
print("=" * 90)

display(
    example_raw
)

print(
    "\nEngineered model input for the same transaction:"
)

display(
    X.head(1).T.rename(
        columns={
            X.index[0]: "Example value"
        }
    )
)

print(
    "\nThis shows how one raw transaction becomes the 19-feature vector."
)

In [ ]:
# ============================================================
# BLOCK 10 — IMPORTANT FEATURE DISTRIBUTIONS
# ============================================================

important_features = [
    "amount",
    "orig_balance_change",
    "dest_balance_change",
    "orig_balance_error",
    "dest_balance_error",
    "log_amount"
]

for feature in important_features:

    plt.figure(
        figsize=(9, 5)
    )

    plt.hist(
        X[feature],
        bins=80
    )

    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.title(
        f"Distribution of {feature}"
    )

    plt.grid(
        axis="y",
        alpha=0.3
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# BLOCK 11 — FEATURE SUMMARY STATISTICS
# ============================================================

summary_stats = X.describe().T

summary_stats["missing"] = X.isna().sum()
summary_stats["unique"] = X.nunique()

print("=" * 90)
print("BLOCK 11 — FEATURE SUMMARY")
print("=" * 90)

display(summary_stats)

In [ ]:
# ============================================================
# BLOCK 12 — TRANSACTION TYPE ENCODING CHECK
# ============================================================

type_summary = pd.DataFrame({
    "Feature": type_columns,
    "Active rows": [
        int(X[col].sum())
        for col in type_columns
    ],
    "Percentage": [
        round(
            float(X[col].mean() * 100),
            4
        )
        for col in type_columns
    ]
})

print("=" * 90)
print("BLOCK 12 — TRANSACTION TYPE ENCODING")
print("=" * 90)

display(type_summary)

print(
    "\nTotal type features:",
    len(type_columns)
)

In [ ]:
# ============================================================
# BLOCK 13 — SAVE FEATURE METADATA
# ============================================================

feature_metadata = {
    "dataset": "PaySim",
    "source_file": str(DATA_PATH),
    "rows_processed": int(len(X)),
    "feature_count": int(len(feature_columns)),
    "feature_columns": list(feature_columns),

    "excluded_columns": [
        "isFraud",
        "isFlaggedFraud",
        "nameOrig",
        "nameDest"
    ],

    "ground_truth_column": "isFraud",

    "transformations": {
        "orig_balance_change":
            "oldbalanceOrg - newbalanceOrig",

        "dest_balance_change":
            "newbalanceDest - oldbalanceDest",

        "orig_balance_error":
            "amount - orig_balance_change",

        "dest_balance_error":
            "amount - dest_balance_change",

        "orig_zero_after":
            "(newbalanceOrig == 0).astype(int)",

        "dest_zero_before":
            "(oldbalanceDest == 0).astype(int)",

        "dest_zero_after":
            "(newbalanceDest == 0).astype(int)",

        "log_amount":
            "np.log1p(amount)",

        "transaction_type":
            "pd.get_dummies(type, prefix='type', dtype=int)"
    },

    "validation": {
        "nan_count": nan_count,
        "inf_count": inf_count
    }
}

metadata_path = (
    OUTPUT_DIR
    / "paysim_feature_engineering_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        feature_metadata,
        file,
        indent=4
    )

print("=" * 90)
print("BLOCK 13 — METADATA SAVED")
print("=" * 90)

print(
    metadata_path.resolve()
)

In [ ]:
# ============================================================
# BLOCK 14 — EXPORT ENGINEERED FEATURES
# ============================================================

features_path = (
    OUTPUT_DIR
    / "paysim_engineered_features_500k.csv"
)

labels_path = (
    OUTPUT_DIR
    / "paysim_ground_truth_500k.csv"
)

# Save model features only.
X.to_csv(
    features_path,
    index=False
)

# Save ground truth separately.
y.to_csv(
    labels_path,
    index=False,
    header=["isFraud"]
)

print("=" * 90)
print("BLOCK 14 — ENGINEERED DATA EXPORTED")
print("=" * 90)

print(
    "Feature matrix:",
    features_path.resolve()
)

print(
    "\nGround truth:",
    labels_path.resolve()
)

print(
    "\nFeature shape:",
    X.shape
)

print(
    "Label shape  :",
    y.shape
)

In [ ]:
# ============================================================
# BLOCK 15 — RELOAD AND VERIFY EXPORTS
# ============================================================

X_reloaded = pd.read_csv(
    features_path
)

y_reloaded = pd.read_csv(
    labels_path
)["isFraud"]

same_shape = (
    X.shape == X_reloaded.shape
)

same_columns = (
    list(X.columns)
    == list(X_reloaded.columns)
)

same_values = np.allclose(
    X.to_numpy(),
    X_reloaded.to_numpy(),
    rtol=1e-5,
    atol=1e-6
)

same_labels = np.array_equal(
    y.to_numpy(),
    y_reloaded.to_numpy()
)

print("=" * 90)
print("BLOCK 15 — EXPORT VERIFICATION")
print("=" * 90)

print("Same shape   :", same_shape)
print("Same columns :", same_columns)
print("Same values  :", same_values)
print("Same labels  :", same_labels)

if not all([
    same_shape,
    same_columns,
    same_values,
    same_labels
]):
    raise ValueError(
        "Export verification failed."
    )

print(
    "\nPASS — exported feature data can be reloaded without change."
)

## Streaming Feature Contract

The later Spark streaming pipeline must reproduce this feature engineering exactly.

### Offline

```text
Raw PaySim
    ↓
Feature Engineering
    ↓
19 features
    ↓
Isolation Forest
```

### Real-time

```text
Kafka JSON transaction
    ↓
Same Feature Engineering
    ↓
Same 19 features
    ↓
Serialized Isolation Forest
```

The following are part of the model contract:

- feature names
- feature formulas
- transaction-type encoding
- feature order
- numerical data types
- excluded columns

A mismatch between training-time and streaming-time features can lead to incorrect model inference.

In [ ]:
# ============================================================
# BLOCK 16 — FINAL REPORT
# ============================================================

print("=" * 90)
print("FINAL FEATURE-ENGINEERING REPORT")
print("=" * 90)

print("\nDataset             :", feature_metadata["dataset"])
print("Rows processed      :", f"{len(X):,}")
print("Feature count       :", len(feature_columns))
print("Ground truth        :", feature_metadata["ground_truth_column"])

print(
    "Excluded columns    :",
    ", ".join(
        feature_metadata["excluded_columns"]
    )
)

print("\nFinal feature order:")
for i, feature in enumerate(
    feature_columns,
    start=1
):
    print(f"{i:02d}. {feature}")

print("\nValidation:")
print("NaN values          :", nan_count)
print("Infinite values     :", inf_count)

print("\nOutputs:")
print("Metadata            :", metadata_path.resolve())
print("Feature CSV         :", features_path.resolve())
print("Ground-truth CSV    :", labels_path.resolve())

print("\nFEATURE ENGINEERING STAGE READY.")